# A/B 验证 · liars_dice1（ACH-SGD vs ACH-Adam）

一键 notebook：配置 → 训练 → 可视化。与 `notebooks/ab_liars_dice1.ipynb`（mirror vs
league）、`notebooks/theta_liars_dice1.ipynb`（theta 扫描）同族——这里固定 **mirror 自
博弈**，只沿**优化器**这一个轴做 A/B。

算法固定为**论文忠实 ACH（theta=1）**，只切**优化器**：论文的恒定学习率 SGD (lr=1e-3) vs Adam 两个学习率（1e-3 严格单因子；3e-4 是 PPO 37-details 给 Adam 的推荐值，对 Adam 更公平但同时动了 lr）。**ACH+Adam 偏离论文 SGD 协议，会如实打印 `ACHFidelityWarning`——这是预期正确行为，不是报错。** 三臂用同一份 `configs/exp/liars_dice1_ach_mlp_mirror.yaml`，只在 notebook 里覆写 optimizer/lr，覆写会写进各臂的 `config.json`（可追溯，AGENTS.md §9）。

- 协议底座：论文忠实 ACH（Fu et al. ICLR 2022），配置直接读 `configs/exp/*.yaml`，
  本 notebook 只覆写 `seed / out_dir / total_env_steps / eval_every_env_steps` 和
  每臂声明的 `overrides`（AGENTS.md §9）。
- 指标：exploitability（exact，OpenSpiel 全树）。eval 是成本大头（约 12 s/点），
  按 `EVAL_EVERY` 而不是 `TOTAL_ENV_STEPS` 预算时间。
- 输出目录：`runs/nb_sgd_adam/liars_dice1/<label>/seed_N/`。**臂按配置 hash 缓存**（不是按目录
  名）：改预算/设备/任一 override 都会被认出来并逐键报告，不会拿旧结果冒充（见参数格
  `ON_STALE`）。
- **SOTA checkpoint**：每臂训练完把评估最优的 checkpoint 复制为 `checkpoints/best/`，
  可直接在 `uv run mjai-play` 里加载试玩。
- 已知脚注：liar's dice 上 ACH 复现本就偏高（~0.32 vs 论文 ~0.17，见
  docs/reproduce_report.md §6）。A/B 看的是**臂之间的相对差**，这个绝对偏差不影响对照，
  但别拿单臂绝对值去对论文。

**直接 Runtime → Run All 即可**；想加深/加宽，改下一格参数后重跑（已完成的臂不会被
重训）。

In [ ]:
# === Parameters ===
GAME        = "liars_dice1"
ARMS = [
    {'label': 'ach_sgd', 'config': 'liars_dice1_ach_mlp_mirror',
     'overrides': {}, 'blurb': 'ACH + 恒定 lr SGD @ 1e-3（论文 H.3）'},
    {'label': 'ach_adam_1e-3', 'config': 'liars_dice1_ach_mlp_mirror',
     'overrides': {'optimizer': 'adam', 'learning_rate': 0.001}, 'blurb': 'ACH + Adam @ 1e-3（会打印 ACHFidelityWarning）'},
    {'label': 'ach_adam_3e-4', 'config': 'liars_dice1_ach_mlp_mirror',
     'overrides': {'optimizer': 'adam', 'learning_rate': 0.0003}, 'blurb': 'ACH + Adam @ 3e-4（会打印 ACHFidelityWarning）'},
]

SEEDS       = [0, 1, 2, 3]   # runs = ARMS x SEEDS
TOTAL_ENV_STEPS = 10000   # per-arm budget (probe depth, not the paper's 1e7)
EVAL_EVERY      = 2500
SHOW_TQDM   = True          # per-arm tqdm bar over env-steps
MAX_TB_POINTS = 2000        # downsample cap for the per-update telemetry read
"""Read-side thinning of the train/* curves for the telemetry panel. The event
files keep full per-update resolution (the blow-ups are intermittent); this
just bounds what the plot has to carry. grad_norm keeps its spikes (bucket
max); 0 disables thinning."""

PROBE_GRAD_NORMS = False    # per-term split; not meaningful here (arms sit at theta 0/1)

ON_STALE    = "error"       # "error" | "retrain" | "skip"
"""What to do with an arm that finished under a DIFFERENT config.

Arms are cached by a fingerprint of their resolved ExperimentConfig, not by
directory name, so raising TOTAL_ENV_STEPS (or changing the device, the eval
cadence, any override) is detected instead of silently reused.

    error    refuse that arm and print which knob changed (nothing deleted).
    retrain  DELETE the arm directory and train it again.
    skip     reuse the mismatched result anyway.
"""

DEVICE      = "cpu"         # "cpu" | "cuda" | None (= whatever the YAML says)
"""CPU is the default on purpose and is the FAST option here: the rollout asks
the policy for ONE decision at a time, so a 21->128->13 forward never fills a
GPU (measured 2809 env-steps/s on CPU vs 441 on CUDA for Liar's Dice). Set
"cuda" only after widening the net/batch enough that the matmul dominates."""

from pathlib import Path
OUT_ROOT = Path("runs/nb_sgd_adam") / GAME
FIG_TITLE = "liars_dice1: ACH-SGD vs ACH-Adam — exploitability vs env-steps (lower = closer to Nash)"

In [ ]:
# === Setup: import the probe machinery (no logic reimplemented here) ===
import sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "tools" / "ab_factor_probe.py").is_file():
    REPO = REPO.parent  # tolerate running from notebooks/
sys.path.insert(0, str(REPO / "tools"))

import ab_factor_probe as ab   # run_arm / arm_status / summarize / render_*
import arm_cache               # config-fingerprint cache (hit / stale / missing)
import policy_view             # final-policy view (rollout + mjai.eval.policy_table)
from IPython.display import Image, display

ORDER  = [a["label"] for a in ARMS]
BLURBS = {a["label"]: a["blurb"] for a in ARMS}

def arm_kwargs(arm):
    return dict(
        overrides=arm["overrides"],
        total_env_steps=TOTAL_ENV_STEPS,
        eval_every_env_steps=EVAL_EVERY,
        root=OUT_ROOT,
        device=DEVICE,
        probe_term_grad_norms=PROBE_GRAD_NORMS,
    )

def train_all():
    statuses, refused = [], []
    for arm in ARMS:
        for seed in SEEDS:
            label = f"{arm['label']:14s} seed={seed}"
            out = ab.arm_dir(OUT_ROOT, arm["label"], seed)
            st = ab.arm_status(arm["label"], arm["config"], seed, **arm_kwargs(arm))
            action, why = arm_cache.resolve(st, ON_STALE, out)
            if action != "train":
                print(f"skip  {label}: {why}", flush=True)
                statuses.append((label, "cached" if action == "skip" else "REFUSED"))
                if action == "refuse":
                    refused.append(label)
                continue
            print(f"train {label}: {why}", flush=True)
            try:
                ab.run_arm(arm["label"], arm["config"], seed, progress_bar=SHOW_TQDM, **arm_kwargs(arm))
                statuses.append((label, "done"))
            except Exception as e:  # keep going; report at the end
                statuses.append((label, f"FAILED: {type(e).__name__}: {e}"))
            print(f"      -> {statuses[-1][1]}", flush=True)
    if refused:
        print()
        print("=" * 72)
        print(f"{len(refused)} arm(s) REFUSED: finished under a different config.")
        print('Set ON_STALE="retrain" to rebuild them or "skip" to reuse them.')
        print("=" * 72)
    return statuses

print(f"{len(ARMS)} arms x {len(SEEDS)} seeds = {len(ARMS) * len(SEEDS)} runs on {DEVICE}")

In [ ]:
# === Train (long cell: arms run sequentially; safe to re-run) ===
statuses = train_all()
for label, st in statuses:
    print(f"{label}: {st}")

In [ ]:
# === Aggregate curves + per-arm results table (incl. SOTA best ckpt) ===
import json

summary = ab.summarize(OUT_ROOT)
for label in ORDER:
    data = summary.get(label)
    if not data:
        print(f"{label:16s} (no data yet)")
        continue
    finals = data.get("final_per_seed", {})
    vals = [round(v, 4) for v in finals.values()]
    mean = round(sum(finals.values()) / len(finals), 4) if finals else None
    print(f"{label:16s} tag={data.get('tag')}  final/seed={vals}  mean={mean}  "
          f"done={len(data.get('done', []))}/{len(finals)}")
print()
print("SOTA best checkpoints (lowest eval point per run, copied to checkpoints/best):")
for bj in sorted(OUT_ROOT.glob("*/seed_*/checkpoints/best/best.json")):
    info = json.loads(bj.read_text(encoding="utf-8"))
    rel = bj.parent.parent.parent.relative_to(OUT_ROOT)
    print(f"  {str(rel):24s} {info['tag']}={info['value']:.4f} @ env_steps={info['env_steps']}")

In [ ]:
# === Comparison figure (mean + min-max band across seeds, one colour per arm) ===
fig_path = ab.render_curves(summary, OUT_ROOT, title=FIG_TITLE, order=ORDER, blurbs=BLURBS)
display(Image(filename=str(fig_path))) if fig_path else print("no curves yet")

In [ ]:
# === Per-update telemetry (downsampled read: grad scale / gate / clip) ===
# One downsampled pass per event file for all tags (tb_eval.read_many_tags),
# so this panel does not re-introduce the full-resolution read the ab_* eval
# curves were fixed to avoid. grad_norm keeps its spikes (bucket max, log axis);
# gate_off_frac has values only on ACH (theta=1) arms, clip_frac only on the
# theta=0 arms -- absent tags render as an empty panel, not a misleading zero.
tel_path = ab.render_telemetry(OUT_ROOT, order=ORDER, max_points=MAX_TB_POINTS)
display(Image(filename=str(tel_path))) if tel_path else print("no telemetry yet")

## 最终策略（每臂学到了什么）

曲线只说「离 Nash 多远」，不说「到底怎么打」。这一格把每臂训练完的策略物化出来。
liar's dice（24576 个信息集）给的是**动作边缘分布 + 按自博弈访问频率排序的 Top-K 信息
集表**，外加完整 CSV 落盘。访问频率来自用该臂自己的策略自博弈 `POLICY_EPISODES` 局，
按 observation 向量 join 回枚举出来的行。

In [ ]:
# === Final policy per arm ===
POLICY_PICK     = "best"   # "best" (SOTA snapshot) | "last" | "step_N"
POLICY_SEED     = 0        # which seed's arm to show
POLICY_EPISODES = 400      # self-play episodes used to rank info states (0 = skip)
POLICY_TOP_K    = 12       # rows in the printed table

import pandas as pd
from mjai.eval.policy_table import PolicyViewError, root_policy, to_records

policy_arms = [(a["label"], ab.arm_dir(OUT_ROOT, a["label"], POLICY_SEED)) for a in ARMS]
policy_arms = [(lab, run) for lab, run in policy_arms if (run / "checkpoints").is_dir()]

if not policy_arms:
    print("no trained arms yet")
else:
    fig_path, views, skipped = policy_view.render_arms(
        policy_arms, OUT_ROOT / "figs" / "policy.png",
        checkpoint=POLICY_PICK, episodes=POLICY_EPISODES, player=0,
    )
    for lab, why in skipped.items():
        print(f"[{lab}] no policy table: {why}")
        try:
            for p, dist in root_policy(dict(policy_arms)[lab], checkpoint=POLICY_PICK).items():
                top = sorted(dist.items(), key=lambda kv: -kv[1])[:5]
                print(f"    root p{p}: " + ", ".join(f"{k}={v:.3f}" for k, v in top))
        except PolicyViewError as e:
            print(f"    (and no policy to load: {e})")
    if fig_path:
        display(Image(filename=str(fig_path)))
    for lab, view in views.items():
        print(f"\n--- {lab} --- {view.checkpoint}")
        rows = view.top_rows(POLICY_TOP_K, player=0)
        df = pd.DataFrame(to_records(view, rows=rows)).set_index("info_state")
        display(df.style.format(precision=3, na_rep="-"))
        csv = OUT_ROOT / "figs" / f"policy_{policy_view.slug(lab)}.csv"
        print(f"    full table ({len(view.labels)} rows) -> "
              f"{policy_view.write_csv(view, csv)}")

## 解读指南

曲线为 mean + min–max 带（n=4 seeds），同预算下谁低谁好；单 seed 发散会把带
撑开，这本身就是信息。

- **`ach_sgd` vs `ach_adam_1e-3`**：严格单因子——只有优化器不同。Adam@1e-3 常常太激进甚至发散，那本身就是结论（ACH 的 SGD 协议不是随手可换的）。
- **`ach_adam_3e-4`**：给 Adam 它自己该用的 lr。若它追平/超过 SGD，说明「ACH 必须配 SGD」更多是 lr 没配对，而非 Adam 本身不行；若仍差，则是优化器本身的差异。
- 遥测格里 `grad_norm`：Adam 会把梯度重标定到 O(1) 附近，SGD 则跟着 ACH 的无界 1/pi_old 抖动——两者不在一个量级是正常的（log 轴）。`clip_frac` 三臂都空（theta=1）。

- **口径**：liar's dice 的 env-step 计所有决策点；mirror 两座都计。想更接近论文预算：把
  `TOTAL_ENV_STEPS` 提到 1e5–1e6 重跑——已完成的臂会因配置 hash 变了被标 stale 并逐键
  报告，按提示把 `ON_STALE` 改成 `"retrain"`（重训）或 `"skip"`（沿用）。
- **产物用法**：每臂 `checkpoints/best/` 是该臂 SOTA 快照，`uv run mjai-play` 会直接列出。